In [1]:
import json
import time
import copy
import random
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT  = Path.home() / "icidea_llm_ids"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"

LABEL_MAP = {0: "NORMAL", 1: "DOS", 2: "FUZZY", 3: "GEAR", 4: "RPM"}
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Load baseline explanations
baseline_df = pd.read_parquet(
    ARTIFACTS_DIR / "section9b_baseline_explanations.parquet"
)
print(f"✓ Loaded {len(baseline_df)} baseline explanations")

# Load windows for frame-level access
windows_df = pd.read_parquet(
    ARTIFACTS_DIR / "section7_windows_with_text.parquet"
)
print(f"✓ Loaded {len(windows_df)} windows")

# Load normal frames for P2 benign insertion
cleaned_df = pd.read_parquet(
    ARTIFACTS_DIR / "section5_cleaned_data.parquet"
)
normal_frames = cleaned_df[cleaned_df["label"] == 0].sample(
    10000, random_state=SEED
).reset_index(drop=True)
print(f"✓ Loaded {len(normal_frames)} normal frames for P2 insertion")

print(f"\nBaseline class distribution:")
print(baseline_df["true_label_name"].value_counts().sort_index().to_string())

✓ Loaded 500 baseline explanations
✓ Loaded 195069 windows
✓ Loaded 10000 normal frames for P2 insertion

Baseline class distribution:
true_label_name
DOS       100
FUZZY     100
GEAR      100
NORMAL    100
RPM       100


In [2]:
# Semantic keyword anchors per attack class
# These define the EXPECTED semantic themes for each class.
# Used in Section 11 to compute semantic alignment scores.
# An explanation that contains none of these keywords has semantically drifted.
# Source: domain knowledge + HCRL dataset characteristics confirmed in Section 5.

SEMANTIC_ANCHORS = {
    "NORMAL": [
        "normal", "steady", "consistent", "periodic", "regular",
        "no anomaly", "baseline", "expected", "routine", "stable",
        "broadcast", "no variation", "legitimate"
    ],
    "DOS": [
        "flood", "flooding", "identical", "repeated", "high frequency",
        "bus overload", "arbitration", "denial", "same frame",
        "constant", "saturate", "overwhelming", "zero payload",
        "dominant", "high rate"
    ],
    "FUZZY": [
        "random", "fuzzy", "unpredictable", "anomalous", "irregular",
        "varied", "diverse", "unknown id", "injection", "chaotic",
        "entropy", "unexpected", "abnormal", "random payload",
        "random id", "variability"
    ],
    "GEAR": [
        "gear", "spoof", "spoofing", "repeated value", "constant payload",
        "ecu", "signal manipulation", "fixed value", "gear position",
        "transmission", "repeated pattern", "consistent data"
    ],
    "RPM": [
        "rpm", "engine speed", "spoof", "spoofing", "repeated",
        "sensor", "mismatch", "constant", "fixed", "engine",
        "rotation", "signal", "repeated pattern", "consistent data"
    ],
}

def compute_alignment_score(explanation: str, label_name: str) -> float:
    """
    Compute semantic alignment score for an explanation.
    
    Score = proportion of anchor keywords found in the explanation.
    Range: 0.0 (no alignment) to 1.0 (all anchors present).
    
    Scientific note: this is a WEAK semantic grounding measure.
    We do NOT claim explanations are "correct" or "wrong."
    We measure whether the explanation uses expected semantic themes
    for the given attack class. Drift = loss of these themes.
    """
    if not explanation:
        return 0.0
    explanation_lower = explanation.lower()
    anchors = SEMANTIC_ANCHORS.get(label_name, [])
    if not anchors:
        return 0.0
    matches = sum(1 for kw in anchors if kw in explanation_lower)
    return matches / len(anchors)

# Verify on baseline
print("BASELINE SEMANTIC ALIGNMENT SCORES")
print("="*50)
for name in ["NORMAL", "DOS", "FUZZY", "GEAR", "RPM"]:
    class_rows = baseline_df[baseline_df["true_label_name"] == name]
    scores = [
        compute_alignment_score(row["qwen_explanation"], name)
        for _, row in class_rows.iterrows()
    ]
    print(f"  {name:7s}  mean={np.mean(scores):.3f}  "
          f"std={np.std(scores):.3f}  "
          f"min={np.min(scores):.3f}  "
          f"max={np.max(scores):.3f}")

BASELINE SEMANTIC ALIGNMENT SCORES
  NORMAL   mean=0.157  std=0.067  min=0.077  max=0.308
  DOS      mean=0.211  std=0.025  min=0.200  max=0.267
  FUZZY    mean=0.129  std=0.019  min=0.125  max=0.250
  GEAR     mean=0.167  std=0.000  min=0.167  max=0.167
  RPM      mean=0.115  std=0.040  min=0.071  max=0.214


In [7]:
SEMANTIC_ANCHORS = {
    "NORMAL": [
        # Words Qwen actually uses for NORMAL
        "normal", "steady", "anomalies", "suspicious",
        "unusual", "malicious", "without", "no indication",
        "no anomaly", "normal operation", "no attack",
        "no signs", "legitimate", "benign", "no malicious",
        "no suspicious", "healthy", "routine",
    ],
    "DOS": [
        # Words Qwen actually uses for DOS
        "identical", "floods", "flooding", "disrupt",
        "attacker", "denial", "service", "attack",
        "packets", "characteristic", "repeated",
        "variations", "identical data", "identical packets",
        "denial of service", "overload", "same frame",
        "uniform", "no variation",
    ],
    "FUZZY": [
        # Words Qwen actually uses for FUZZY
        "variability", "random", "unpredictability",
        "fuzzing", "variations", "frequent changes",
        "intentionally", "robustness", "payloads",
        "introduce", "chaotic", "diverse", "irregular",
        "anomalous", "unpredictable", "high variability",
        "randomness", "erratic",
    ],
    "GEAR": [
        # Words Qwen actually uses for GEAR
        "gear", "position", "sensor", "repeated",
        "same content", "value", "monitoring",
        "controlling", "sending", "same", "consistent",
        "spoof", "spoofing", "fixed value",
        "repeated pattern", "constant", "static",
    ],
    "RPM": [
        # Words Qwen actually uses for RPM
        "repeated", "identical", "same", "sensor",
        "sending", "repetition", "structure",
        "characteristic", "payload", "value",
        "consistent", "constant", "reading",
        "same structure", "same content",
        "repeated multiple", "fixed",
    ],
}

def compute_alignment_score(explanation: str, label_name: str) -> float:
    """
    Compute semantic alignment score.
    Score = proportion of anchor keywords found in explanation.
    Range: 0.0 to 1.0.

    Scientific framing: measures presence of expected semantic themes.
    Does NOT assess factual correctness.
    Drift = loss of class-appropriate semantic themes after perturbation.
    """
    if not explanation:
        return 0.0
    explanation_lower = explanation.lower()
    anchors = SEMANTIC_ANCHORS.get(label_name, [])
    if not anchors:
        return 0.0
    matches = sum(1 for kw in anchors if kw in explanation_lower)
    return matches / len(anchors)

# Verify on baseline
print("BASELINE SEMANTIC ALIGNMENT SCORES (vocabulary-grounded anchors)")
print("="*65)
for name in ["NORMAL", "DOS", "FUZZY", "GEAR", "RPM"]:
    class_rows = baseline_df[baseline_df["true_label_name"] == name]
    scores = [
        compute_alignment_score(row["qwen_explanation"], name)
        for _, row in class_rows.iterrows()
    ]
    print(f"  {name:7s}  mean={np.mean(scores):.3f}  "
          f"std={np.std(scores):.3f}  "
          f"min={np.min(scores):.3f}  "
          f"max={np.max(scores):.3f}")

# Keyword match detail
print(f"\nKEYWORD MATCH DETAIL (sample per class):")
print("="*65)
for name in ["NORMAL", "DOS", "FUZZY", "GEAR", "RPM"]:
    sample_exp = baseline_df[
        baseline_df["true_label_name"] == name
    ].iloc[0]["qwen_explanation"].lower()
    matched = [kw for kw in SEMANTIC_ANCHORS[name] if kw in sample_exp]
    missed  = [kw for kw in SEMANTIC_ANCHORS[name] if kw not in sample_exp]
    print(f"\n  {name}:")
    print(f"    Matched ({len(matched)}): {matched}")
    print(f"    Missed  ({len(missed)}):  {missed[:6]}...")

BASELINE SEMANTIC ALIGNMENT SCORES (vocabulary-grounded anchors)
  NORMAL   mean=0.291  std=0.063  min=0.167  max=0.444
  DOS      mean=0.623  std=0.020  min=0.579  max=0.632
  FUZZY    mean=0.539  std=0.069  min=0.167  max=0.556
  GEAR     mean=0.647  std=0.000  min=0.647  max=0.647
  RPM      mean=0.562  std=0.029  min=0.529  max=0.588

KEYWORD MATCH DETAIL (sample per class):

  NORMAL:
    Matched (6): ['normal', 'steady', 'anomalies', 'unusual', 'malicious', 'without']
    Missed  (12):  ['suspicious', 'no indication', 'no anomaly', 'normal operation', 'no attack', 'no signs']...

  DOS:
    Matched (12): ['identical', 'floods', 'disrupt', 'attacker', 'denial', 'service', 'attack', 'packets', 'characteristic', 'variations', 'identical data', 'no variation']
    Missed  (7):  ['flooding', 'repeated', 'identical packets', 'denial of service', 'overload', 'same frame']...

  FUZZY:
    Matched (10): ['variability', 'random', 'unpredictability', 'fuzzing', 'variations', 'frequent chan

In [6]:
from collections import Counter
import re

def get_top_words(explanations, top_n=30, min_length=4):
    """Extract most frequent meaningful words from a list of explanations."""
    stopwords = {
        "this", "that", "with", "from", "have", "been", "they", "their",
        "there", "which", "will", "would", "could", "should", "about",
        "into", "also", "such", "than", "more", "each", "when", "where",
        "what", "these", "those", "some", "data", "frame", "frames",
        "sequence", "telemetry", "traffic", "network", "can", "bus",
        "indicates", "suggest", "suggests", "pattern", "patterns",
        "behavior", "typical", "vehicle", "through", "within", "across",
        "between", "while", "other", "further", "however", "therefore",
        "overall", "analysis", "indicating", "consistent"
    }
    word_counts = Counter()
    for exp in explanations:
        if not exp:
            continue
        words = re.findall(r'\b[a-z]{4,}\b', exp.lower())
        for w in words:
            if w not in stopwords:
                word_counts[w] += 1
    return word_counts.most_common(top_n)

print("TOP WORDS IN QWEN EXPLANATIONS PER CLASS")
print("="*60)
print("(These are the words Qwen actually uses — anchors must match these)")
print()

for name in ["NORMAL", "DOS", "FUZZY", "GEAR", "RPM"]:
    exps = baseline_df[
        baseline_df["true_label_name"] == name
    ]["qwen_explanation"].tolist()
    top_words = get_top_words(exps, top_n=25)
    print(f"  {name}:")
    print(f"    {[w for w, c in top_words]}")
    print()

TOP WORDS IN QWEN EXPLANATIONS PER CLASS
(These are the words Qwen actually uses — anchors must match these)

  NORMAL:
    ['shows', 'anomalies', 'normal', 'activity', 'suspicious', 'gear', 'steady', 'denial', 'service', 'represent', 'appears', 'without', 'unusual', 'communication', 'fuzzy', 'detected', 'malicious', 'state', 'logic', 'indications', 'injection', 'manipulation', 'operation', 'fuzzing', 'changes']

  DOS:
    ['identical', 'consists', 'entirely', 'packets', 'additional', 'information', 'characteristic', 'denial', 'service', 'attack', 'attacker', 'floods', 'messages', 'disrupt', 'normal', 'operations', 'variations', 'repeated', 'variation', 'system']

  FUZZY:
    ['shows', 'high', 'level', 'variability', 'random', 'unpredictability', 'system', 'characteristic', 'variations', 'frequent', 'changes', 'following', 'protocol', 'fuzzing', 'attacks', 'attackers', 'intentionally', 'introduce', 'communication', 'test', 'robustness', 'security', 'measures', 'payloads', 'standard']

In [8]:
def perturb_p1_payload_mutation(frames: list, seed=42) -> tuple:
    """
    P1: Payload Byte Mutation

    Scientific purpose: Tests whether a minimal data-layer change
    destabilizes explanation semantics while preserving classification.

    For DOS/GEAR/RPM (zero-variance payloads), any byte change is
    maximally detectable as a deviation from the attack pattern.
    For FUZZY/NORMAL, the change blends into existing variance.

    Implementation: mutate one byte in the middle frame by ±1.
    Wraps at 0/255 boundaries.
    """
    rng = random.Random(seed)
    perturbed = copy.deepcopy(frames)

    target_idx = len(frames) // 2
    frame = perturbed[target_idx]
    byte_idx = rng.randint(0, 7)
    original_val = frame["data"][byte_idx]
    delta = rng.choice([-1, 1])
    new_val = (original_val + delta) % 256
    frame["data"][byte_idx] = new_val

    return perturbed, {
        "frame_idx": target_idx,
        "byte_idx":  byte_idx,
        "original":  original_val,
        "mutated":   new_val,
        "delta":     delta,
    }


def perturb_p2_benign_insertion(frames: list, normal_frames_df,
                                  seed=42) -> tuple:
    """
    P2: Benign Frame Insertion

    Scientific purpose: Tests whether injecting one legitimate normal
    traffic frame into an attack window causes Qwen to contextually
    reframe its explanation — context pollution.

    The inserted frame is real normal traffic (from Section 5),
    making the perturbation protocol-compliant and realistic.
    Window size is preserved by dropping the last frame.
    """
    perturbed = copy.deepcopy(frames)

    normal_row = normal_frames_df.sample(1, random_state=seed).iloc[0]
    insert_frame = {
        "timestamp": perturbed[len(perturbed)//2]["timestamp"],
        "can_id":    int(normal_row["can_id"]),
        "dlc":       int(normal_row["dlc"]),
        "data":      [int(normal_row[f"data{i}"]) for i in range(8)],
    }

    insert_pos = len(perturbed) // 2
    perturbed.insert(insert_pos, insert_frame)
    perturbed = perturbed[:len(frames)]  # maintain window size

    return perturbed, {
        "insert_pos":    insert_pos,
        "inserted_id":   insert_frame["can_id"],
        "inserted_data": insert_frame["data"],
    }


def perturb_p3_frame_reorder(frames: list, seed=42) -> tuple:
    """
    P3: Frame Reordering

    Scientific purpose: Tests temporal reasoning stability.
    Swapping two adjacent frames changes the apparent sequence of
    events without altering any payload content. If Qwen's explanation
    changes, it demonstrates sensitivity to temporal ordering — a sign
    of temporal reasoning instability.

    Implementation: swap adjacent frames at the window midpoint.
    """
    perturbed = copy.deepcopy(frames)
    swap_idx = len(frames) // 2
    perturbed[swap_idx], perturbed[swap_idx + 1] = \
        perturbed[swap_idx + 1], perturbed[swap_idx]

    return perturbed, {
        "swap_idx_a": swap_idx,
        "swap_idx_b": swap_idx + 1,
    }


def perturb_p4_confidence_reduction(text: str, securebert_pred: str,
                                      securebert_conf: float) -> tuple:
    """
    P4: Prompt Confidence Reduction

    Scientific purpose: Tests whether Qwen's explanation changes when
    the meta-information (SecureBERT confidence) is reduced, even
    though the telemetry is IDENTICAL. Pure prompt-layer perturbation.

    If the explanation changes (hedges, shifts themes, loses
    assertiveness), it reveals that Qwen is influenced by classifier
    confidence — explanation instability driven by meta-context,
    not data.

    Implementation: replace confidence 1.000 with 0.623
    (deliberately non-round to appear realistic).
    """
    perturbed_conf = 0.623
    return text, securebert_pred, perturbed_conf, {
        "original_conf":  securebert_conf,
        "perturbed_conf": perturbed_conf,
    }


def perturb_p5_semantic_steering(text: str, securebert_pred: str,
                                   securebert_conf: float,
                                   label_name: str) -> tuple:
    """
    P5: Semantic Steering

    Scientific purpose: The most novel perturbation type.
    Adds a steering instruction to the prompt that directs Qwen toward
    an ALTERNATIVE reasoning theme without changing the telemetry or
    classification.

    If Qwen follows the steering and shifts its explanation theme,
    this demonstrates that LLM explanations are not purely grounded
    in telemetry data but are sensitive to prompt framing.

    This is NOT adversarial prompting to break the system.
    It is a controlled test of explanation groundedness.

    Steering instructions are class-specific:
    - DOS    → steer toward timing irregularity (away from flooding)
    - FUZZY  → steer toward payload anomaly (away from ID randomness)
    - GEAR   → steer toward frequency anomaly (away from value spoofing)
    - RPM    → steer toward timing anomaly (away from signal spoofing)
    - NORMAL → steer toward subtle anomaly detection
    """
    STEERING_INSTRUCTIONS = {
        "DOS":    "Note: Pay particular attention to inter-frame timing "
                  "irregularities and arrival interval patterns.",
        "FUZZY":  "Note: Focus your analysis on payload byte value "
                  "distributions and data content anomalies.",
        "GEAR":   "Note: Pay particular attention to frame transmission "
                  "frequency and inter-arrival timing patterns.",
        "RPM":    "Note: Focus your analysis on inter-frame timing "
                  "consistency and transmission interval regularity.",
        "NORMAL": "Note: Look carefully for any subtle deviations from "
                  "expected periodic broadcast patterns.",
    }

    steering = STEERING_INSTRUCTIONS.get(label_name, "")

    return text, securebert_pred, securebert_conf, steering, {
        "steering_instruction": steering,
        "target_label":         label_name,
    }


def frames_to_text(frames: list) -> str:
    """Reconstruct telemetry text from frame list (Section 7 format)."""
    n = len(frames)
    lines = [f"CAN Bus Telemetry Sequence ({n} frames):"]
    for i, frame in enumerate(frames, start=1):
        can_id_hex = f"{frame['can_id']:04X}"
        dlc = frame["dlc"]
        data_bytes = frame["data"][:dlc]
        data_hex = " ".join(f"{b:02X}" for b in data_bytes)
        ts = frame["timestamp"]
        lines.append(
            f"[{i:03d}] T={ts:.3f} ID={can_id_hex} "
            f"DLC={dlc} DATA={data_hex}"
        )
    return "\n".join(lines)


print("✓ All 5 perturbation functions defined")
print()
print("  P1: Payload byte mutation   (±1, middle frame, data layer)")
print("  P2: Benign frame insertion  (real normal traffic, context layer)")
print("  P3: Frame reordering        (adjacent swap, temporal layer)")
print("  P4: Confidence reduction    (1.000 → 0.623, prompt layer)")
print("  P5: Semantic steering       (class-specific redirect, prompt layer)")

✓ All 5 perturbation functions defined

  P1: Payload byte mutation   (±1, middle frame, data layer)
  P2: Benign frame insertion  (real normal traffic, context layer)
  P3: Frame reordering        (adjacent swap, temporal layer)
  P4: Confidence reduction    (1.000 → 0.623, prompt layer)
  P5: Semantic steering       (class-specific redirect, prompt layer)


In [11]:
# Load classifier results to get window_start_ts for frame lookup
classifier_results = pd.read_parquet(
    ARTIFACTS_DIR / "section9a_classifier_results.parquet"
)

# Build lookup: window_idx → window_start_ts
idx_to_ts = classifier_results["window_start_ts"].to_dict()
idx_to_text = classifier_results["text"].to_dict()

print(f"✓ Loaded classifier results for frame lookup")
print(f"  Classifier results rows: {len(classifier_results)}")
print(f"  Columns: {list(classifier_results.columns)}")
print(f"\n  baseline_df window_idx sample:    {baseline_df['window_idx'].head().tolist()}")
print(f"  classifier_results index sample:  {list(idx_to_ts.keys())[:5]}")

# Check overlap
baseline_idxs = set(baseline_df["window_idx"].tolist())
classifier_idxs = set(idx_to_ts.keys())
overlap = baseline_idxs & classifier_idxs
print(f"\n  baseline_df unique window_idx:    {len(baseline_idxs)}")
print(f"  classifier_results index keys:    {len(classifier_idxs)}")
print(f"  Overlapping keys:                 {len(overlap)}")

# Now attempt frame lookup for first row
first_idx = baseline_df["window_idx"].iloc[0]
ts = idx_to_ts.get(first_idx)
print(f"\n  First window_idx: {first_idx}")
print(f"  Mapped timestamp: {ts}")

if ts is not None:
    match = windows_df[windows_df["window_start_ts"] == ts]
    print(f"  windows_df match rows: {len(match)}")
    if len(match) > 0:
        frames = json.loads(match.iloc[0]["frames_json"])
        print(f"  Frames in window: {len(frames)}")
        print(f"  ✓ Frame lookup chain works end-to-end")
    else:
        print(f"  ⚠ No match in windows_df for this timestamp")
else:
    print(f"  ⚠ window_idx not found in classifier_results index")

✓ Loaded classifier results for frame lookup
  Classifier results rows: 1500
  Columns: ['text', 'label', 'label_name', 'pred_label', 'pred_name', 'confidence', 'correct', 'prob_NORMAL', 'prob_DOS', 'prob_FUZZY', 'prob_GEAR', 'prob_RPM', 'frames_json', 'window_start_ts']

  baseline_df window_idx sample:    [0, 1, 2, 3, 4]
  classifier_results index sample:  [0, 1, 2, 3, 4]

  baseline_df unique window_idx:    500
  classifier_results index keys:    1500
  Overlapping keys:                 500

  First window_idx: 0
  Mapped timestamp: 1479121529.933034
  windows_df match rows: 1
  Frames in window: 14
  ✓ Frame lookup chain works end-to-end


In [10]:
print("baseline_df columns:")
print(list(baseline_df.columns))
print()
print("windows_df columns:")
print(list(windows_df.columns))

baseline_df columns:
['window_idx', 'true_label', 'true_label_name', 'securebert_pred', 'securebert_pred_name', 'securebert_confidence', 'qwen_raw', 'qwen_classification', 'qwen_confidence', 'qwen_explanation', 'qwen_key_indicators', 'qwen_temporal', 'json_valid', 'parse_error', 'inference_time_s']

windows_df columns:
['frames_json', 'label', 'window_start_ts', 'window_end_ts', 'text']


In [12]:
perturbed_records = []
skipped = 0

for _, row in baseline_df.iterrows():
    window_idx = row["window_idx"]

    # Step 1: window_idx → window_start_ts via classifier_results
    ts = idx_to_ts.get(window_idx)
    if ts is None:
        skipped += 1
        continue

    # Step 2: window_start_ts → frames_json via windows_df
    match = windows_df[windows_df["window_start_ts"] == ts]
    if len(match) == 0:
        skipped += 1
        continue
    frames = json.loads(match.iloc[0]["frames_json"])

    # Step 3: get original text from classifier_results
    original_text = idx_to_text.get(window_idx, row.get("text", ""))

    label_name = row["true_label_name"]

    # Baseline semantic alignment score
    baseline_alignment = compute_alignment_score(
        row["qwen_explanation"], label_name
    )

    base_record = {
        "window_idx":              window_idx,
        "true_label":              row["true_label"],
        "true_label_name":         label_name,
        "securebert_pred":         row["securebert_pred"],
        "securebert_pred_name":    row["securebert_pred_name"],
        "securebert_confidence":   row["securebert_confidence"],
        "baseline_explanation":    row["qwen_explanation"],
        "baseline_classification": row["qwen_classification"],
        "baseline_confidence":     row["qwen_confidence"],
        "baseline_temporal":       row["qwen_temporal"],
        "baseline_indicators":     row["qwen_key_indicators"],
        "baseline_alignment":      baseline_alignment,
        "frames_json":             json.dumps(frames),
    }

    # --- P1: Payload mutation ---
    p1_frames, p1_meta = perturb_p1_payload_mutation(frames, seed=SEED)
    perturbed_records.append({
        **base_record,
        "perturbation_type":          "P1_PAYLOAD",
        "perturbed_text":             frames_to_text(p1_frames),
        "securebert_conf_for_prompt": row["securebert_confidence"],
        "steering_instruction":       "",
        "perturbation_meta":          json.dumps(p1_meta),
    })

    # --- P2: Benign insertion ---
    p2_frames, p2_meta = perturb_p2_benign_insertion(
        frames, normal_frames, seed=SEED
    )
    perturbed_records.append({
        **base_record,
        "perturbation_type":          "P2_INSERTION",
        "perturbed_text":             frames_to_text(p2_frames),
        "securebert_conf_for_prompt": row["securebert_confidence"],
        "steering_instruction":       "",
        "perturbation_meta":          json.dumps(p2_meta),
    })

    # --- P3: Frame reorder ---
    p3_frames, p3_meta = perturb_p3_frame_reorder(frames, seed=SEED)
    perturbed_records.append({
        **base_record,
        "perturbation_type":          "P3_REORDER",
        "perturbed_text":             frames_to_text(p3_frames),
        "securebert_conf_for_prompt": row["securebert_confidence"],
        "steering_instruction":       "",
        "perturbation_meta":          json.dumps(p3_meta),
    })

    # --- P4: Confidence reduction ---
    _, _, p4_conf, p4_meta = perturb_p4_confidence_reduction(
        original_text, row["securebert_pred_name"],
        row["securebert_confidence"]
    )
    perturbed_records.append({
        **base_record,
        "perturbation_type":          "P4_PROMPT",
        "perturbed_text":             original_text,
        "securebert_conf_for_prompt": p4_conf,
        "steering_instruction":       "",
        "perturbation_meta":          json.dumps(p4_meta),
    })

    # --- P5: Semantic steering ---
    _, _, p5_conf, p5_steering, p5_meta = perturb_p5_semantic_steering(
        original_text, row["securebert_pred_name"],
        row["securebert_confidence"], label_name
    )
    perturbed_records.append({
        **base_record,
        "perturbation_type":          "P5_STEERING",
        "perturbed_text":             original_text,
        "securebert_conf_for_prompt": p5_conf,
        "steering_instruction":       p5_steering,
        "perturbation_meta":          json.dumps(p5_meta),
    })

perturbed_df = pd.DataFrame(perturbed_records)

print(f"✓ Built perturbed dataset")
print(f"  Total records: {len(perturbed_df):,}")
print(f"  Skipped:       {skipped}")
print(f"\nPerturbation type distribution:")
print(perturbed_df["perturbation_type"].value_counts().to_string())
print(f"\nClass × perturbation type:")
print(pd.crosstab(
    perturbed_df["true_label_name"],
    perturbed_df["perturbation_type"]
).to_string())
print(f"\nBaseline alignment scores (carried into perturbed dataset):")
print(perturbed_df.groupby("true_label_name")["baseline_alignment"]
      .mean().round(3).to_string())

✓ Built perturbed dataset
  Total records: 2,500
  Skipped:       0

Perturbation type distribution:
perturbation_type
P1_PAYLOAD      500
P2_INSERTION    500
P3_REORDER      500
P4_PROMPT       500
P5_STEERING     500

Class × perturbation type:
perturbation_type  P1_PAYLOAD  P2_INSERTION  P3_REORDER  P4_PROMPT  P5_STEERING
true_label_name                                                                
DOS                       100           100         100        100          100
FUZZY                     100           100         100        100          100
GEAR                      100           100         100        100          100
NORMAL                    100           100         100        100          100
RPM                       100           100         100        100          100

Baseline alignment scores (carried into perturbed dataset):
true_label_name
DOS       0.623
FUZZY     0.539
GEAR      0.647
NORMAL    0.291
RPM       0.562


In [13]:
print("PERTURBATION VERIFICATION")
print("="*60)

# P1: Confirm byte mutation on DOS
p1_dos = perturbed_df[
    (perturbed_df["true_label_name"] == "DOS") &
    (perturbed_df["perturbation_type"] == "P1_PAYLOAD")
].iloc[0]
meta = json.loads(p1_dos["perturbation_meta"])
print(f"P1 DOS mutation:")
print(f"  Frame {meta['frame_idx']}, byte {meta['byte_idx']}: "
      f"{meta['original']} → {meta['mutated']} (delta={meta['delta']})")
print(f"  Perturbed line {meta['frame_idx']+1}: "
      f"{p1_dos['perturbed_text'].split(chr(10))[meta['frame_idx']+1]}")

# P2: Confirm benign insertion
p2_dos = perturbed_df[
    (perturbed_df["true_label_name"] == "DOS") &
    (perturbed_df["perturbation_type"] == "P2_INSERTION")
].iloc[0]
meta2 = json.loads(p2_dos["perturbation_meta"])
print(f"\nP2 DOS benign insertion:")
print(f"  Inserted at position {meta2['insert_pos']}, "
      f"CAN ID={meta2['inserted_id']:04X}")
print(f"  Inserted line: "
      f"{p2_dos['perturbed_text'].split(chr(10))[meta2['insert_pos']+1]}")

# P4: Confirm confidence reduction
p4_sample = perturbed_df[
    perturbed_df["perturbation_type"] == "P4_PROMPT"
].iloc[0]
meta4 = json.loads(p4_sample["perturbation_meta"])
print(f"\nP4 confidence reduction:")
print(f"  Original confidence:  {meta4['original_conf']}")
print(f"  Perturbed confidence: {meta4['perturbed_conf']}")

# P5: Confirm steering instructions per class
print(f"\nP5 steering instructions per class:")
for name in ["NORMAL", "DOS", "FUZZY", "GEAR", "RPM"]:
    sample = perturbed_df[
        (perturbed_df["true_label_name"] == name) &
        (perturbed_df["perturbation_type"] == "P5_STEERING")
    ].iloc[0]
    print(f"  {name:7s}: '{sample['steering_instruction'][:70]}...'")

PERTURBATION VERIFICATION
P1 DOS mutation:
  Frame 7, byte 1: 14 → 13 (delta=-1)
  Perturbed line 8: [008] T=1479121917.848 ID=0440 DLC=8 DATA=FF 0D 00 00 FF 00 01 53

P2 DOS benign insertion:
  Inserted at position 7, CAN ID=0130
  Inserted line: [008] T=1479121917.848 ID=0130 DLC=8 DATA=8D 7E 00 FF 2C 80 03 D0

P4 confidence reduction:
  Original confidence:  0.9999911785125732
  Perturbed confidence: 0.623

P5 steering instructions per class:
  NORMAL : 'Note: Look carefully for any subtle deviations from expected periodic ...'
  DOS    : 'Note: Pay particular attention to inter-frame timing irregularities an...'
  FUZZY  : 'Note: Focus your analysis on payload byte value distributions and data...'
  GEAR   : 'Note: Pay particular attention to frame transmission frequency and int...'
  RPM    : 'Note: Focus your analysis on inter-frame timing consistency and transm...'


In [14]:
artifact_path = ARTIFACTS_DIR / "section10_perturbed_dataset.parquet"
perturbed_df.to_parquet(artifact_path, index=False)
size_mb = artifact_path.stat().st_size / 1e6

print(f"✓ Saved perturbed dataset")
print(f"  Path:    {artifact_path}")
print(f"  Size:    {size_mb:.1f} MB")
print(f"  Records: {len(perturbed_df):,}")
print(f"  Columns: {list(perturbed_df.columns)}")
print()
print("NEXT STEPS:")
print("1. Upload section10_perturbed_dataset.parquet to Kaggle dataset")
print("   (add to existing icidea-llm-ids-benchmark dataset)")
print("2. Run Kaggle inference notebook for perturbed explanations")
print("3. Download results back to Mac for Section 11 analysis")

✓ Saved perturbed dataset
  Path:    /Users/deepakpatnaik/icidea_llm_ids/artifacts/section10_perturbed_dataset.parquet
  Size:    0.4 MB
  Records: 2,500
  Columns: ['window_idx', 'true_label', 'true_label_name', 'securebert_pred', 'securebert_pred_name', 'securebert_confidence', 'baseline_explanation', 'baseline_classification', 'baseline_confidence', 'baseline_temporal', 'baseline_indicators', 'baseline_alignment', 'frames_json', 'perturbation_type', 'perturbed_text', 'securebert_conf_for_prompt', 'steering_instruction', 'perturbation_meta']

NEXT STEPS:
1. Upload section10_perturbed_dataset.parquet to Kaggle dataset
   (add to existing icidea-llm-ids-benchmark dataset)
2. Run Kaggle inference notebook for perturbed explanations
3. Download results back to Mac for Section 11 analysis
